<a href="https://colab.research.google.com/github/sodiq-classic/Academic_Projects/blob/main/Sodiq_text_embedding_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
%cd /content/drive/MyDrive/Desktop/RRC/TERM_3/Neural_Network

/content/drive/MyDrive/Desktop/RRC/TERM_3/Neural_Network


In [29]:
!pip install autocorrect

In [30]:
import nltk
from nltk.corpus import gutenberg
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from autocorrect import Speller
import re
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

# Download necessary nltk resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [31]:
# Ensure the necessary resources are downloaded
nltk.download('gutenberg')

# Load the three plays using nltk.corpus.gutenberg.raw
hamlet = gutenberg.raw('shakespeare-hamlet.txt').lower()
macbeth = gutenberg.raw('shakespeare-macbeth.txt').lower()
julius_caesar = gutenberg.raw('shakespeare-caesar.txt').lower()

# Combine the texts into a single variable
shakespeare = hamlet + macbeth + julius_caesar

# Display the first few characters to verify the output
shakespeare[:500]

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


"[the tragedie of hamlet by william shakespeare 1599]\n\n\nactus primus. scoena prima.\n\nenter barnardo and francisco two centinels.\n\n  barnardo. who's there?\n  fran. nay answer me: stand & vnfold\nyour selfe\n\n   bar. long liue the king\n\n   fran. barnardo?\n  bar. he\n\n   fran. you come most carefully vpon your houre\n\n   bar. 'tis now strook twelue, get thee to bed francisco\n\n   fran. for this releefe much thankes: 'tis bitter cold,\nand i am sicke at heart\n\n   barn. haue you had quiet guard?\n  fran. not"

In [32]:
# Initialize tools
speller = Speller(lang='en')
stop_words = set(stopwords.words('english'))  # Load the standard stopwords
lemmatizer = WordNetLemmatizer()
#Tokenize the text into sentences and then into words
sentences = sent_tokenize(shakespeare)
tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]


Lemmatization was  chosen because unlike stemming, which just cuts off prefixes or suffixes, lemmatization considers the context of the word to derive the meaningful root.

In [33]:
# create a manual stop words to remove from the texts
manual_stopwords = ['aboutus', 'ist', 'ile', 'vse', 'oft', 'sha', 'dec', 'wit',
                    'pol', 'cal']
stop_words.update(manual_stopwords)

In [34]:
#Spell correction and clean up each word in each sentence
def clean_and_process(sentence):
    processed_sentence = []
    for word in sentence:
        word = word.lower()  # Convert to lowercase
        word = speller(word)  # Correct spelling mistakes
        word = re.sub(r'[^a-zA-Z]', '', word)  # Remove non-alphabetic characters
        word = re.sub(r'\b\w{1,2}\b', '', word) # Remove 1 & 2 letters
        word = re.sub(r'\s+', '', word).strip() # Remove space
        if word and word not in stop_words:  # Remove stopwords
            word = lemmatizer.lemmatize(word)  # Lemmatize the word
            processed_sentence.append(word)
    return processed_sentence

In [35]:
# Process each sentence
processed_sentences = [clean_and_process(sentence) for sentence in \
                       tokenized_sentences]

In [36]:
# Print out the words in the first five sentences of the processed text
processed_sentences[:10]

[['tragedy', 'hamlet', 'william', 'shakespeare', 'act', 'prime'],
 ['scene', 'prima'],
 ['enter', 'bernard', 'francisco', 'two', 'sentinel'],
 ['bernard'],
 [],
 ['fran'],
 ['nay', 'answer', 'stand', 'unfold', 'self', 'bar'],
 ['long', 'like', 'king', 'fran'],
 ['bernard'],
 ['bar']]

In [37]:
# Cleaned and tokenized data would already be prepared in 'processed_sentences_no_speller'

#Train a CBOW Word2Vec model
cbow_model = Word2Vec(
    sentences=processed_sentences,  # Tokenized and cleaned sentences
    vector_size=100,  # Size of word embeddings
    window=5,         # Context window size
    min_count=3,      # Ignore words with frequency lower than 3
    sg=0,             # 0 means CBOW, 1 means Skip-gram
    epochs=10         # Number of iterations over the corpus
)

#Get the 20 most frequent words and their counts
most_frequent_words = cbow_model.wv.key_to_index  # Dictionary of words and their frequency rank
top_20_words = list(most_frequent_words.items())[:20]  # Get top 20 most frequent words

#Print word and count using get_vecattr to access 'count'
for word, value in top_20_words:
    count = cbow_model.wv.get_vecattr(word, "count")  # Access word count
    print(f"Word: {word}, Count: {count}")


Word: ham, Count: 337
Word: thou, Count: 307
Word: lord, Count: 306
Word: shall, Count: 300
Word: come, Count: 284
Word: king, Count: 248
Word: enter, Count: 230
Word: good, Count: 221
Word: let, Count: 220
Word: mac, Count: 205
Word: thy, Count: 202
Word: like, Count: 200
Word: cesar, Count: 193
Word: one, Count: 188
Word: make, Count: 185
Word: know, Count: 184
Word: thee, Count: 174
Word: self, Count: 166
Word: would, Count: 163
Word: von, Count: 159


In [38]:
# Step 1: Create and train the Skip-gram Word2Vec model
skipgram_model = Word2Vec(
    sentences=processed_sentences,  # Tokenized and cleaned sentences
    vector_size=100,  # Size of word embeddings
    window=5,         # Context window size
    min_count=3,      # Ignore words with frequency lower than 3
    sg=1,             # 1 means Skip-gram (sg=0 for CBOW)
    epochs=10         # Number of training epochs
)

In [39]:
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

# Convert GloVe to Word2Vec format
glove_input_file = "glove.6B.100d.txt"
word2vec_output_file = "glove.6B.100d.word2vec.txt"  # Temporary file for Word2Vec format
glove2word2vec(glove_input_file, word2vec_output_file)  # Convert

# Load the GloVe model (now in Word2Vec format)
glove_model = KeyedVectors.load_word2vec_format(word2vec_output_file, binary=False)

<ipython-input-39-acd0b8c26b92>:7: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, word2vec_output_file)  # Convert


In [40]:
# Function to get the 5 most similar words
def print_most_similar(model, word):
    try:
        similar_words = model.wv.most_similar(word, topn=5)
        print(f"\nMost similar words to '{word}':")
        for sim_word, similarity in similar_words:
            print(f"{sim_word}: {similarity:.4f}")
    except KeyError:
        print(f"'{word}' not in vocabulary")

In [41]:
# Example usage for each model
print("CBOW Model:")
print_most_similar(cbow_model, 'hamlet')
print_most_similar(cbow_model, 'cauldron')
print_most_similar(cbow_model, 'nature')
print_most_similar(cbow_model, 'spirit')
print_most_similar(cbow_model, 'general')
print_most_similar(cbow_model, 'prythee')

CBOW Model:

Most similar words to 'hamlet':
alert: 0.9995
king: 0.9995
seems: 0.9994
soul: 0.9994
within: 0.9994

Most similar words to 'cauldron':
hand: 0.9985
god: 0.9984
blood: 0.9984
like: 0.9984
without: 0.9984

Most similar words to 'nature':
great: 0.9998
put: 0.9998
time: 0.9998
self: 0.9997
like: 0.9997

Most similar words to 'spirit':
like: 0.9997
self: 0.9997
whose: 0.9997
soul: 0.9997
make: 0.9997

Most similar words to 'general':
heaven: 0.9996
even: 0.9996
since: 0.9996
whose: 0.9996
call: 0.9996

Most similar words to 'prythee':
life: 0.9986
deed: 0.9986
faire: 0.9985
though: 0.9985
nature: 0.9985


In [42]:
print("Skip-gram Model:")
print_most_similar(skipgram_model, 'hamlet')
print_most_similar(skipgram_model, 'cauldron')
print_most_similar(skipgram_model, 'nature')
print_most_similar(skipgram_model, 'spirit')
print_most_similar(skipgram_model, 'general')
print_most_similar(skipgram_model, 'prythee')

Skip-gram Model:

Most similar words to 'hamlet':
alert: 0.9867
queen: 0.9833
colony: 0.9800
ham: 0.9783
rosincrance: 0.9775

Most similar words to 'cauldron':
sway: 0.9980
secret: 0.9979
beside: 0.9978
element: 0.9978
creature: 0.9977

Most similar words to 'nature':
innocent: 0.9915
score: 0.9914
action: 0.9912
kind: 0.9909
offence: 0.9908

Most similar words to 'spirit':
wound: 0.9931
couldst: 0.9927
enough: 0.9925
dost: 0.9924
comst: 0.9922

Most similar words to 'general':
suffer: 0.9977
barbara: 0.9976
sooner: 0.9975
broad: 0.9975
strife: 0.9975

Most similar words to 'prythee':
dist: 0.9979
caution: 0.9979
oldest: 0.9978
assure: 0.9977
saying: 0.9977


In [43]:
# Function to get the 5 most similar words
def print_most_similar(model, word):
    try:
        similar_words = model.most_similar(word, topn=5)
        print(f"\nMost similar words to '{word}':")
        for sim_word, similarity in similar_words:
            print(f"{sim_word}: {similarity:.4f}")
    except KeyError:
        print(f"'{word}' not in vocabulary")

In [44]:
print("\nPretrained GloVe Model:")
print_most_similar(glove_model, 'hamlet')
print_most_similar(glove_model, 'cauldron')
print_most_similar(glove_model, 'nature')
print_most_similar(glove_model, 'spirit')
print_most_similar(glove_model, 'general')
print_most_similar(glove_model, 'prythee')


Pretrained GloVe Model:

Most similar words to 'hamlet':
village: 0.6999
town: 0.6559
situated: 0.5926
located: 0.5661
unincorporated: 0.5599

Most similar words to 'cauldron':
caldron: 0.7603
flame: 0.6907
lit: 0.5912
torch: 0.5582
candle: 0.5477

Most similar words to 'nature':
natural: 0.7198
true: 0.7150
aspects: 0.7124
life: 0.7035
view: 0.6961

Most similar words to 'spirit':
passion: 0.7443
faith: 0.7213
love: 0.6864
sense: 0.6724
devotion: 0.6692

Most similar words to 'general':
secretary: 0.7607
chief: 0.7243
gen.: 0.6899
president: 0.6798
vice: 0.6727
'prythee' not in vocabulary


For the glove model, prythee is not found in the vocabulary

In [45]:
# Function to calculate cosine similarity between two terms in a model
def cosine_similarity(model, word1, word2):
    try:
        similarity = model.wv.similarity(word1, word2)
        print(f"Similarity between '{word1}' and '{word2}': {similarity:.4f}")
    except KeyError:
        print(f"One or both terms ('{word1}', '{word2}') not in the vocabulary.")

# Word pairs to compare
word_pairs = [
    ('brutus', 'murder'),
    ('lady macbeth', 'queen gertrude'),
    ('fortinbras', 'norway'),
    ('rome', 'norway'),
    ('ghost', 'spirit'),
    ('macbeth', 'hamlet')
]

In [46]:
# For each model, calculate cosine similarity for each pair
print("\nCBOW Model:")
for word1, word2 in word_pairs:
    cosine_similarity(cbow_model, word1, word2)


CBOW Model:
One or both terms ('brutus', 'murder') not in the vocabulary.
One or both terms ('lady macbeth', 'queen gertrude') not in the vocabulary.
Similarity between 'fortinbras' and 'norway': 0.9993
Similarity between 'rome' and 'norway': 0.9992
Similarity between 'ghost' and 'spirit': 0.9987
Similarity between 'macbeth' and 'hamlet': 0.9985


In [47]:
print("\nSkip-gram Model:")
for word1, word2 in word_pairs:
    cosine_similarity(skipgram_model, word1, word2)


Skip-gram Model:
One or both terms ('brutus', 'murder') not in the vocabulary.
One or both terms ('lady macbeth', 'queen gertrude') not in the vocabulary.
Similarity between 'fortinbras' and 'norway': 0.9970
Similarity between 'rome' and 'norway': 0.9879
Similarity between 'ghost' and 'spirit': 0.9766
Similarity between 'macbeth' and 'hamlet': 0.8722


In [48]:
# Function to calculate cosine similarity between two terms in a model
def cosine_similarity_glove(model, word1, word2):
    try:
        similarity = model.similarity(word1, word2)
        print(f"Similarity between '{word1}' and '{word2}': {similarity:.4f}")
    except KeyError:
        print(f"One or both terms ('{word1}', '{word2}') not in the vocabulary.")

In [49]:
print("\nPretrained GloVe Model:")
for word1, word2 in word_pairs:
    cosine_similarity_glove(glove_model, word1, word2)


Pretrained GloVe Model:
Similarity between 'brutus' and 'murder': 0.0736
One or both terms ('lady macbeth', 'queen gertrude') not in the vocabulary.
Similarity between 'fortinbras' and 'norway': -0.0290
Similarity between 'rome' and 'norway': 0.2858
Similarity between 'ghost' and 'spirit': 0.4282
Similarity between 'macbeth' and 'hamlet': 0.4294


**Shakespeare CBOW and Skip-gram models:** These models capture the specific literary context of the words in Shakespeare’s works, especially where names of characters or places are concerned. They are specialized, and this reflect in higher similarity scores for Shakespearean terms.

**GloVe:** As a general-purpose model, GloVe will likely perform better on common, modern terms like ghost and spirit, or countries like Rome and Norway, but struggle with character names like Brutus or Lady Macbeth that are domain-specific to Shakespeare.

In [50]:
# Function to calculate the most similar words based on linear combinations of word vectors
def most_similar_combination(model, positive_terms, negative_terms=None):
    try:
        similar_words = model.wv.most_similar(positive=positive_terms, negative=negative_terms, topn=5)
        print(f"\nMost similar words to '{positive_terms}' with negatives '{negative_terms}':")
        for sim_word, similarity in similar_words:
            print(f"{sim_word}: {similarity:.4f}")
    except KeyError as e:
        print(f"Word not found in vocabulary: {e}")

# Word vector combinations
combinations = [
    (['denmark', 'queen'], None),
    (['scotland', 'army', 'general'], None),
    (['father', 'woman'], ['man']),
    (['mother', 'man'], ['woman'])
]

In [51]:
# Perform similarity search for each model
print("CBOW Model:")
for positive, negative in combinations:
    most_similar_combination(cbow_model, positive, negative)

CBOW Model:

Most similar words to '['denmark', 'queen']' with negatives 'None':
soul: 0.9997
king: 0.9997
great: 0.9996
seems: 0.9996
could: 0.9996

Most similar words to '['scotland', 'army', 'general']' with negatives 'None':
like: 0.9995
either: 0.9995
would: 0.9995
even: 0.9995
death: 0.9995

Most similar words to '['father', 'woman']' with negatives '['man']':
reason: 0.9988
either: 0.9988
bloody: 0.9988
put: 0.9988
body: 0.9988

Most similar words to '['mother', 'man']' with negatives '['woman']':
spirit: 0.9989
look: 0.9989
nature: 0.9989
dead: 0.9989
love: 0.9989


In [52]:
print("Skip-gram Model:")
for positive, negative in combinations:
    most_similar_combination(skipgram_model, positive, negative)

Skip-gram Model:

Most similar words to '['denmark', 'queen']' with negatives 'None':
ophelia: 0.9941
rosincrance: 0.9935
colony: 0.9933
exit: 0.9931
ratio: 0.9929

Most similar words to '['scotland', 'army', 'general']' with negatives 'None':
sooner: 0.9986
audience: 0.9985
bond: 0.9985
frankly: 0.9985
glory: 0.9984

Most similar words to '['father', 'woman']' with negatives '['man']':
hamlet: 0.9547
lost: 0.9541
alert: 0.9530
think: 0.9517
mother: 0.9511

Most similar words to '['mother', 'man']' with negatives '['woman']':
look: 0.9733
stand: 0.9720
dreadful: 0.9712
clown: 0.9707
forth: 0.9707


In [53]:
# Function to calculate the most similar words based on linear combinations of word vectors
def most_similar_combinations(model, positive_terms, negative_terms=None):
    try:
        similar_words = model.most_similar(positive=positive_terms, negative=negative_terms, topn=5)
        print(f"\nMost similar words to '{positive_terms}' with negatives '{negative_terms}':")
        for sim_word, similarity in similar_words:
            print(f"{sim_word}: {similarity:.4f}")
    except KeyError as e:
        print(f"Word not found in vocabulary: {e}")

In [54]:
print("Pretrained GloVe Model:")
for positive, negative in combinations:
    most_similar_combinations(glove_model, positive, negative)

Pretrained GloVe Model:

Most similar words to '['denmark', 'queen']' with negatives 'None':
sweden: 0.7462
norway: 0.7017
kingdom: 0.6879
princess: 0.6800
britain: 0.6786

Most similar words to '['scotland', 'army', 'general']' with negatives 'None':
force: 0.7447
british: 0.7336
military: 0.7317
command: 0.7294
forces: 0.7241

Most similar words to '['father', 'woman']' with negatives '['man']':
mother: 0.9025
daughter: 0.8675
wife: 0.8535
husband: 0.8279
grandmother: 0.8112

Most similar words to '['mother', 'man']' with negatives '['woman']':
father: 0.8926
brother: 0.8531
son: 0.8221
uncle: 0.8052
friend: 0.8045


**Shakespeare CBOW/Skip-gram Models:**

These models are gave more specific, contextually relevant results for Shakespearean themes, such as terms related to characters or roles in the plays. For example, combinations involving Scotland and army should evoke characters like Macbeth or Duncan, and combinations like denmark + queen should result in Shakespearean references like Gertrude.

For abstract combinations like father - man + woman, they seem to return gendered roles seen in the context of Shakespeare's plays.

**Pretrained GloVe Model:**

GloVe performs better for combinations involving modern relationships between gender roles and parental terms, such as father - man + woman. It is trained on general-purpose text, so it handles these combinations in a broader, more modern context.

However, for Shakespeare-specific combinations like denmark + queen or scotland + army + general, it lacks the detailed domain knowledge of Shakespeare's works and provide more general results related to royalty, nations, and military roles without the specific connections to Shakespearean characters.

## Overall Comment
**Model Performance Summary:**

**Shakespeare CBOW:** Captures general word relationships in Shakespeare's text but struggles with nuanced or rare words.

**Shakespeare Skip-gram:** Better at handling rare terms and context-specific meanings but noisier with frequent words.

**Pretrained GloVe:** Good for modern, common word associations, but lacks domain-specific understanding of Shakespearean language.

**To build a better Shakespearean word embedding model:**

Use Shakespeare’s complete works, Elizabethan literature, annotated texts, historical dictionaries, and literary criticism.
Incorporate performance scripts for dialogue understanding.
Consider larger embeddings and contextual models like BERT fine-tuned on this data for better word relationships in Shakespearean English.